In [1]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import numpy as np
import os
import re

In [2]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [3]:
def tokenize_lematize(tekst):
    doc = nlp(tekst)
    
    izbrane_besede = []
    
    # želimo odstraniti osebe, kraji, jeziki, narodi...
    odstrani_pos = ['PROPN', 'PRON', 'VERB', 'ADV']
    odstrani_entitete = {'PERSON', 'GPE', 'LOC', 'NORP', 'FAC', 'ORG'}
    
    #mnozica prepoznanih entitet
    for token in doc:
        # odstranimo i
        if token.pos_ in odstrani_pos:
            continue
        if token.ent_type_ in odstrani_entitete:
            continue
            
        # spacy ima oznake NOUN, ADJ
        if token.pos_ in ['NOUN', 'ADJ']:
            beseda = token.lemma_.lower()
            # samo crke in dolžina nad 2 znaka
            if beseda.isalpha() and len(beseda) > 2:
                izbrane_besede.append(beseda)
                
    return izbrane_besede

In [4]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()


custom_words = {
    # založniški šum 
    'book', 'novel', 'story', 'author', 'literature', 'edition', 'seller', 
    'read', 'reader', 'page', 'chapter', 'write', 'writer', 'publish', 
    'publication', 'review', 'times', 'york', 'print', 'copy', 'best', 
    'original', 'classic', 'introduction', 'note', 'debut', 'thriller',
    'unique', 'fascinating', 'scandal', 'major', 'character', 'cover', 'magazine',
    'self', 'series', 'volume', 'masterpiece', 'translation', 'film', 'tale', 'course',
    
    # splošni opisi 
    'way', 'thing', 'important', 'practical', 'young', 'boy', 'girl', 
    'human', 'people', 'great', 'good', 'bad', 'true', 'new', 'old',
    'life', 'world', 'everything', 'day', 'time', 'year', 'make',
    'take', 'come', 'think', 'feel', 'know', 'look', 'want', 'large', 'small',
    'man', 'woman', 'literary', 'secret', 'isbn', 'mother', 'sister', 'father',
    'little', 'room', 'place', 'end', 'first', 'second', 'beautiful', 'family',
    'friend', 'brother',
    
    #iz izpisa
    'professional', 'guide', 'experience', 'natural', 'vivid', 'narrative',
    'compelling', 'extraordinary', 'powerful', 'voice', 'mind'
}


all_stopwords = list(base_stopwords.union(custom_words))

In [ ]:
metadata= pd.read_csv(r'C:\Users\mokro\Desktop\diploma\leto_2003\knjige_03_metadata.csv')
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\leto_2003\03_ang_opisi\*.txt')[:150]

In [6]:
def extract_num(filename):
    match = re.search(r'opis_(\d+)', filename)
    return int(match.group(1)) if match else None

In [22]:
metadata['book_id'] = metadata['Knjiga'].apply(lambda x: int(re.search(r'(\d+)', str(x)).group(1)) if re.search(r'(\d+)', str(x)) else None)
id_to_title = dict(zip(metadata['book_id'], metadata['Originalni naslov']))
metadata_new = metadata.drop_duplicates(subset=['book_id'])

In [25]:
vectorizer= CountVectorizer(stop_words= all_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=3, 
                            max_df=0.7)

In [26]:
X = vectorizer.fit_transform(filepaths) 

c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [27]:
def nmf(X, k, max_iter=500, tol=1e-4, random_state=42):
    """
    Nenegativna matrična faktorizacija, ki uporablja pravila za posodobitev elementov na podlagi množenja.

    Parametri:
    -----------
    X : ndarray (m x n)
        Nenegativna matrika
    k : int
        Stevilo komponent (teme/ žanri)
    max_iter : int
        Maksimalno število iteracij
    tol : float
        Toleranca konvergence (izračunano s pomočjo reconstruction error)
    random_state : int
        

    Vrne:
    --------
    W : ndarray (m x k)
    H : ndarray (k x n)
    errors : list
        Reconstruction za vsako iteracijo
    """
    
    np.random.seed(random_state)
    
    m, n = X.shape
    
    #zacnemo z nakljucnimi nenegativnimi vrednostmi
    W = np.random.rand(m, k)
    H = np.random.rand(k, n)
    
    eps = 1e-9
    errors = []
    
    for i in range(max_iter):
        
        # posodabljanje H
        H *= (W.T @ X) / (W.T @ W @ H + eps) # + eps, da ne delimo z 0
        # posodabljanje W
        W *= (X @ H.T) / (W @ (H @ H.T) + eps)
        
        # reconstruction error
        error = np.linalg.norm(X - W @ H, 'fro')
        errors.append(error)
        
        # konvergenca
        if i > 0 and abs(errors[-2] - error) < tol:
            break

    return W, H, errors

In [28]:
k = 4
W, H, _ = nmf(X, k)

In [30]:
dominant_topics = np.argmax(W, axis=1) + 1
filenames = [os.path.basename(f) for f in filepaths]

results = []
for i in range(len(filenames)):
    book_id = extract_num(filenames[i])
    title = id_to_title.get(book_id, "Title Not Found")
    results.append({
        'Filename': filenames[i],
        'ID': book_id,
        'Naslov': title,
        'Tema': dominant_topics[i]
    })

df_final = pd.DataFrame(results)


feature_names = vectorizer.get_feature_names_out()

for t in range(1, k + 1):
    print(f"\n--- TEMA {t} ---")
    
    # top besede
    top_words = [feature_names[i] for i in H[t-1].argsort()[-10:]]
    print(f"Besede: {', '.join(top_words)}")
    
    # seznam knjig
    
    titles_in_topic = df_final[df_final['Tema'] == t]['Naslov'].tolist()
    print(f"Naslovi knjig: {titles_in_topic}")
    print(len(titles_in_topic))


--- TEMA 1 ---
Besede: town, art, wedding, search, prison, power, marriage, heart, city, love
Naslovi knjig: ['The Full Cupboard of Life', 'The Tail of Emily Windsnap', 'The Kite Runner', 'What Was She Thinking? [Notes on a Scandal]', 'Oryx and Crake', 'Shantaram', 'The Namesake', 'Persepolis: The Story of a Childhood', 'Eleven Minutes', 'The Wedding', 'Three Wishes', 'This is Where I Leave You', 'Between Sisters', 'Blankets', 'True Believer', 'Boy Meets Boy', 'Blue Like Jazz: Nonreligious Thoughts on Christian Spirituality', 'Second Glance', "The Queen's Fool", 'The Birth of Venus', 'The Second Summer of the Sisterhood', 'The Will to Change: Men, Masculinity, and Love', "The Time Traveler's Wife", 'Sex, Drugs, and Cocoa Puffs: A Low Culture Manifesto', 'Quicksilver', 'Summer People', "Trickster's Choice", 'Princess in Pink', 'Brick Lane']
29

--- TEMA 2 ---
Besede: knowledge, matter, dead, simple, beginning, career, paper, order, habit, creative
Naslovi knjig: ['Harry Potter and the 